In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import gradio as gr


In [19]:
pip install gradio

In [3]:
with open("/content/t8.shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:100])

This is the 100th Etext file presented by Project Gutenberg, and
is presented in cooperation with Wo


In [4]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

print(chars)
print("Vocabulary Size:", vocab_size)

['\n', ' ', '!', '"', '#', '%', '&', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '|', '}', '~']
Vocabulary Size: 91


In [5]:
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for i, ch in enumerate(chars)}

encoded_text = [char2idx[ch] for ch in text]

print("Sample text:", text[:100])
print("Encoded sample:", encoded_text[:100])
print("Vocab size:", len(chars))

Sample text: This is the 100th Etext file presented by Project Gutenberg, and
is presented in cooperation with Wo
Encoded sample: [51, 69, 70, 80, 1, 70, 80, 1, 81, 69, 66, 1, 16, 15, 15, 81, 69, 1, 36, 81, 66, 85, 81, 1, 67, 70, 73, 66, 1, 77, 79, 66, 80, 66, 75, 81, 66, 65, 1, 63, 86, 1, 47, 79, 76, 71, 66, 64, 81, 1, 38, 82, 81, 66, 75, 63, 66, 79, 68, 11, 1, 62, 75, 65, 0, 70, 80, 1, 77, 79, 66, 80, 66, 75, 81, 66, 65, 1, 70, 75, 1, 64, 76, 76, 77, 66, 79, 62, 81, 70, 76, 75, 1, 84, 70, 81, 69, 1, 54, 76]
Vocab size: 91


In [6]:


seq_length = 50

inputs = []
targets = []

for i in range(len(encoded_text) - seq_length):

    input_seq = encoded_text[i : i + seq_length]
    target_seq = encoded_text[i + 1 : i + seq_length + 1]

    inputs.append(input_seq)
    targets.append(target_seq)

X = torch.tensor(inputs)
Y = torch.tensor(targets)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nSample Input:", X[0])
print("Sample Target:", Y[0])

X shape: torch.Size([5458149, 50])
Y shape: torch.Size([5458149, 50])

Sample Input: tensor([51, 69, 70, 80,  1, 70, 80,  1, 81, 69, 66,  1, 16, 15, 15, 81, 69,  1,
        36, 81, 66, 85, 81,  1, 67, 70, 73, 66,  1, 77, 79, 66, 80, 66, 75, 81,
        66, 65,  1, 63, 86,  1, 47, 79, 76, 71, 66, 64, 81,  1])
Sample Target: tensor([69, 70, 80,  1, 70, 80,  1, 81, 69, 66,  1, 16, 15, 15, 81, 69,  1, 36,
        81, 66, 85, 81,  1, 67, 70, 73, 66,  1, 77, 79, 66, 80, 66, 75, 81, 66,
        65,  1, 63, 86,  1, 47, 79, 76, 71, 66, 64, 81,  1, 38])


In [7]:
class LSTMModel(nn.Module):

    def __init__(self, vocab_size, embed_size=128, hidden_size=256, num_layers=2):
        super(LSTMModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)

        self.lstm = nn.LSTM(
            embed_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, vocab_size)


    def forward(self, x, hidden=None):

        x = self.embedding(x)

        out, hidden = self.lstm(x, hidden)

        out = out.reshape(out.size(0) * out.size(1), out.size(2))

        out = self.fc(out)

        return out, hidden

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = TensorDataset(X, Y)

dataloader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True,
    num_workers=0,
    pin_memory=True if device.type == "cuda" else False
)

In [10]:
model = LSTMModel(vocab_size=vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.to(device)

for epoch in range(2):

    loop = tqdm(dataloader, desc="Training")

    for X_batch, Y_batch in loop:

        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        optimizer.zero_grad()

        outputs, _ = model(X_batch)

        loss = criterion(outputs, Y_batch.view(-1))

        loss.backward()

        optimizer.step()

        loop.set_postfix(loss=loss.item())

Training: 100%|██████████| 21321/21321 [10:26<00:00, 34.01it/s, loss=1.1]


In [26]:
model.eval()
model.to(device)

# safe encoding
def safe_encode(text):
    return [char2idx.get(c, 0) for c in text]


def generate_text(start_text, temperature, length):

    input_seq = torch.tensor([safe_encode(start_text)]).to(device)

    hidden = None
    output_text = start_text

    for _ in range(length):

        out, hidden = model(input_seq, hidden)

        out = out[:, -1, :] / temperature

        probs = F.softmax(out, dim=1)

        idx = torch.multinomial(probs, 1).item()

        output_text += idx2char[idx]

        input_seq = torch.tensor([[idx]]).to(device)

    return output_text


# UI
interface = gr.Interface(
    fn=generate_text,
    inputs=[
        gr.Textbox(label="Start Text", value="to be"),
        gr.Slider(0.1, 2.0, value=0.8, label="Temperature"),
        gr.Slider(50, 500, value=200, label="Length")
    ],
    outputs=gr.Textbox(label="Generated Text"),
    title="🔥 LSTM Text Generator",
    description="Generate text using a trained character-level LSTM model"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://155c24488b439c1ee9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
